In [3]:
# !uv add requests beautifulsoup4 scikit-learn

In [1]:
import requests
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
access_token = "YOUR_ACCESS_TOKEN"

In [ ]:
def read_onenote(access_token):

    headers = {
        "Authorization": f"Bearer {access_token}"
    }

    base_url = "https://graph.microsoft.com/v1.0/me/onenote"
    documents = []

    # Get all sections
    sections_response = requests.get(
        f"{base_url}/sections",
        headers=headers
    )

    sections_response.raise_for_status()
    sections = sections_response.json()["value"]

    for section in sections:

        section_id = section["id"]
        section_name = section["displayName"]

        # Get pages belonging to this section
        pages_response = requests.get(
            f"{base_url}/sections/{section_id}/pages",
            headers=headers
        )

        pages_response.raise_for_status()
        pages = pages_response.json()["value"]

        for page in pages:

            page_id = page["id"]
            page_title = page["title"]

            # Get actual page content
            content_response = requests.get(
                f"{base_url}/pages/{page_id}/content",
                headers=headers
            )

            content_response.raise_for_status()
            page_html = content_response.text

            # Convert OneNote HTML into searchable text
            soup = BeautifulSoup(page_html, "html.parser")

            page_text = soup.get_text(
                separator=" ",
                strip=True
            )

            document = {
                "id": page_id,
                "title": page_title,
                "section": section_name,
                "content": page_text,
                "source_type": "onenote",
                "last_modified": page.get("lastModifiedDateTime")
            }

            documents.append(document)

    return documents

In [ ]:
documents = read_onenote(access_token)
len(documents)

In [ ]:
documents[0]